In [1]:
import json
import numpy as np
import pandas as pd
from langdetect import detect, detect_langs
from copy import deepcopy
import unicodedata
import ast

In [2]:
with open("data/all_results.json", "r", encoding="utf-8") as f:
            survey_full_results = json.load(f)

In [3]:
# 1. Setup the Master Answer Key
# Combining both languages into one set for high-speed, order-independent lookup
attn_chk_pass_answer_DK = [
    'En person, som ikke er fra et politisk parti, men støtter et parti.',
    "En person fra NGO'er, der fremmer valgdeltagelse."
]
attn_chk_pass_answer_EN = [
    'Someone who is not from a political party but supports a party.',
    'Someone from NGOs promoting electoral participation.'
]
valid_answers = set(attn_chk_pass_answer_DK + attn_chk_pass_answer_EN)

In [4]:
# 2. Initialize counters and lists
fail_count = 0
pass_count = 0
no_pass_list = []

# 3. Process the results
for res in survey_full_results:
    # Safely retrieve the list of answers from the nested dictionary
    attn_chk_res = res['surveyData']['votingExperience']['ATTN_CHK']
    if detect(attn_chk_res[0]) == 'en':  # output: 'en' for english; 'da' for danish
        res['surveyData']['language'] = 'EN'
    else: 
        res['surveyData']['language'] = 'DA'
    
    # Calculate overlap using set intersection
    # This finds which of the respondent's answers are in our 'valid_answers' set
    correct_selections = set(attn_chk_res) & valid_answers
    num_correct = len(correct_selections)
    total_selected = len(attn_chk_res)

    # 1. PASS: Exactly 2 correct and ONLY 2 selected.
    if num_correct == 2 and total_selected == 2:
        res['ATTN_CHK_PF'] = 'PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        pass_count += 1
        
    # 2. HALF_PASS: Picked 1 or 2 correct, but total selections must be less than 4.
    # This keeps people who picked (Correct + Incorrect) or (Correct + Correct + Incorrect).
    elif num_correct >= 1 and total_selected < 4:
        res['ATTN_CHK_PF'] = 'HALF_PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        no_pass_list.append(attn_chk_res)
        
    # 3. FAIL: Either zero correct OR they "gamed" the system by picking 4+ options.
    else:
        res['ATTN_CHK_PF'] = 'FAIL'
        # Optional: Add a specific reason for your internal tracking
        if total_selected >= 4:
            res['ATTN_CHK_FAIL_REASON'] = 'Over-selection (4+ items)'
        else:
            res['ATTN_CHK_FAIL_REASON'] = 'Zero correct matches'
            
        no_pass_list.append(attn_chk_res)
        fail_count += 1

# 4. Optional: Print summary
print(f"Results processed: {pass_count} Pass, {fail_count} Fail.")

Results processed: 557 Pass, 395 Fail.


In [5]:
column_name_list = []
initial_survey_data = survey_full_results[0]['surveyData']
for col_nam in initial_survey_data.keys():
    if type(initial_survey_data[col_nam]) is dict:
        column_name_list = column_name_list + list(initial_survey_data[col_nam].keys())
    else: column_name_list.append(col_nam)
column_name_list = column_name_list + list(survey_full_results[0].keys())[-2:]

In [6]:
translation_dict = {
    'SCREEN_ELIGIBILITY': {"EN": ['Yes'],
              "DA": ['Ja']},
    'SCREEN_RESIDENCE': {"EN": ['Yes'],
              "DA": ['Ja']},
    'ALCL11': {"EN": ['Municipal tax in Aarhus Municipality must be raised, and the money must be spent on better welfare.', 'It is possible to save money in the public sector without affecting public welfare.',
                     'More tasks in the public sector must be solved by private companies.', 'Aarhus Municipality must make it cheaper to run a business.',
                     'Aarhus Municipality must prioritize that school pupils are mixed according to ethnicity and social background.', 'The politicians must prevent the construction of mosques.',
                     'More parking spaces must be established in Aarhus Municipality.', 'The city council’s temporary stop for the expansion of Aarhus Harbor must be made permanent.',
                     'Car traffic in Aarhus city center must be limited, for example through one-way directions, speed reductions and a zero-emission zone.',
                     'The municipality must continue with the plans for the new football stadium in Kongelunden, even if the costs rise again.'],
              "DA": ['Kommuneskatten i Aarhus Kommune skal hæves, og pengene skal bruges på bedre velfærd.', 'Det er muligt at spare penge i den offentlige sektor uden at påvirke den offentlige velfærd.',
                     'Flere opgaver i den offentlige sektor skal løses af private virksomheder.', 'Aarhus Kommune skal gøre det billigere at drive virksomhed.',
                     'Aarhus Kommune skal prioritere, at skoleelever blandes på tværs af etnicitet og social baggrund.', 'Politikerne skal forhindre opførelsen af moskéer.',
                     'Der skal etableres flere parkeringspladser i Aarhus Kommune.', 'Byrådets midlertidige stop for udvidelsen af Aarhus Havn skal gøres permanent.',
                     'Biltrafikken i Aarhus centrum skal begrænses, for eksempel gennem ensretninger, hastighedsnedsættelser og en nulemissionszone.',
                     'Kommunen skal fortsætte planerne om det nye fodboldstadion i Kongelunden, selv hvis omkostningerne stiger igen.']},
    'VOTE_LOC': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'VOTE_NAT': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'ATTN_CHK': {"EN": ['Someone from a political party.', 'Someone who is not from a political party but supports a party.',
                       'Someone from NGOs promoting electoral participation.', 'No one from a political party.'],
              "DA": ['En person fra et politisk parti.', 'En person, som ikke er fra et politisk parti, men støtter et parti.',
                    "En person fra NGO'er, der fremmer valgdeltagelse.", 'Ingen fra et politisk parti.']},
    'GENDER': {"EN": ['Male', 'Female', 'Non-binary', 'Prefer not to say'],
              "DA": ['Mand', 'Kvinde', 'Non-binær', 'Ønsker ikke at svare']},
    'EDUCATION': {"EN": ['Primary and lower secondary education (e.g., Folkeskole or Friskole)', 'Upper secondary education (e.g., STX, HTX, HHX, HF)',
                        'Vocational education and training (EUD or EUX)', 'Short-cycle higher education (1-2 years)', 'Medium-length higher education (3-4 years)',
                        'Long-cycle higher education (5-7 years)', 'PhD or other research degree.', 'PhD or other research degree', 'Other', 'Prefer not to say'],
              "DA": ['Grundskole eller tilsvarende (f.eks. Folkeskole eller Friskole)', 'Gymnasial uddannelse (f.eks. STX, HTX, HHX, HF)',
                    'Erhvervsuddannelse (f.eks. EUD eller EUX)', 'Kort videregående uddannelse (1-2år)',
                    'Mellemlang videregående uddannelse (3–4 år)', 'Lang videregående uddannelse (5–7 år)',
                    'PhD anden forskeruddannelse', 'PhD anden forskeruddannelse', 'Andet', 'Ønsker ikke at svare']},
    'JOB': {"EN": ['In paid work (employee, self-employed, working for your family business) or temporary absent.', 'In education (not paid for by employer), even if on vacation',
                  'Unemployed and actively looking for a job', 'Unemployed, wanting a job but not actively looking for a job', 'Permanently sick or disabled',
                  'Retired', 'In community or military service', 'Doing housework, looking after children or other persons', 'Other', 'Prefer not to say'],
              "DA": ['I lønnet arbejde (ansat, selvstændig eller i familiens virksomhed), også midlertidigt fraværende', 
                     'Under uddannelse (ikke betalt af arbejdsgiver), også hvis du har haft ferie', 'Arbejdsløs og aktivt jobsøgende',	
                     'Arbejdsløs, ønsker job, søger ikke aktivt', 'Varigt syg eller med handicap', 'Pensioneret',
                     'I samfunds- eller militærtjeneste', 'Hjemmegående/passer børn eller andre personer', 'Andet', 'Ønsker ikke at svare']},
    'SRQV1': {"EN": ['It should only be allowed to give one vote to each candidate.', 'It should be allowed to give maximally two or three votes to one.',
              'It should be allowed to give more votes to one candidate, but an additional vote should "cost" more than one vote.', 
              'It should be allowed to give all votes to just one candidate without additional "costs".'],
              "DA": ['Det bør kun være tilladt at give én stemme til hver kandidat.', 'Det bør være tilladt at give højst to eller tre stemmer til én kandidat.',
                     'Det bør være tilladt at give flere stemmer til én kandidat, men en ekstra stemme bør "koste" mere end én stemme.',
                     'Det bør være tilladt at give alle stemmer til én kandidat uden ekstra "omkostninger".']}
}

In [7]:
survey_df_dict = {col_nam: [] for col_nam in column_name_list}

for res in survey_full_results:
    for col_nam in column_name_list:
        if "ATTN_CHK_" in col_nam:
            survey_df_dict[col_nam].append(res[col_nam])
        elif col_nam == 'SRQV1':
            survey_df_dict[col_nam].append(res['srqv1'])
        else:
            if col_nam in res['surveyData'].keys():
                survey_df_dict[col_nam].append(res['surveyData'][col_nam])
            else:
                for key in res['surveyData'].keys():
                    if type(res['surveyData'][key]) is dict and col_nam in list(res['surveyData'][key].keys()):
                        survey_df_dict[col_nam].append(res['surveyData'][key][col_nam])

In [8]:
# Translate survey responses to English at the item level.
# Important: do NOT use respondent-level language as the condition for translation.
# Some respondents have mixed-language values, e.g. English ATTN_CHK but Danish ALCL11.
# The rule below is: Danish labels are translated; English labels stay unchanged;
# blanks/NaN/unexpected free-text values are preserved.

def _norm_label(x):
    """Normalize unicode and whitespace for safer dictionary matching."""
    if not isinstance(x, str):
        return x
    return " ".join(unicodedata.normalize("NFKC", x).split())


def _make_translation_maps(translation_dict):
    translation_maps = {}
    for col, langs in translation_dict.items():
        da_vals = langs["DA"]
        en_vals = langs["EN"]

        if len(da_vals) != len(en_vals):
            raise ValueError(
                f"Translation list length mismatch for {col}: "
                f"{len(da_vals)} DA values vs {len(en_vals)} EN values"
            )

        mapping = {}
        for da, en in zip(da_vals, en_vals):
            # DA -> EN
            mapping[da] = en
            mapping[_norm_label(da)] = en
            # EN -> EN, so the translation is safe to run repeatedly
            mapping[en] = en
            mapping[_norm_label(en)] = en

        translation_maps[col] = mapping
    return translation_maps


def _translate_value(value, mapping):
    if isinstance(value, list):
        return [_translate_value(v, mapping) for v in value]
    if pd.isna(value) or value == "":
        return value
    return mapping.get(value, mapping.get(_norm_label(value), value))


survey_df_dict_ENG = deepcopy(survey_df_dict)
translation_maps = _make_translation_maps(translation_dict)

for trans_col_nam, mapping in translation_maps.items():
    survey_df_dict_ENG[trans_col_nam] = [
        _translate_value(value, mapping)
        for value in survey_df_dict_ENG[trans_col_nam]
    ]

# QC: this should be empty if all dictionary-covered Danish labels were translated.
remaining_danish_labels = {}
for trans_col_nam, langs in translation_dict.items():
    da_labels = set(langs["DA"]) | {_norm_label(x) for x in langs["DA"]}
    bad_rows = []
    for i, value in enumerate(survey_df_dict_ENG[trans_col_nam]):
        values = value if isinstance(value, list) else [value]
        if any(isinstance(v, str) and (v in da_labels or _norm_label(v) in da_labels) for v in values):
            bad_rows.append(i)
    if bad_rows:
        remaining_danish_labels[trans_col_nam] = bad_rows

print("Remaining Danish dictionary labels after translation:", remaining_danish_labels)


Remaining Danish dictionary labels after translation: {}


In [9]:
# Create analysis-ready dummy variables for multi-select questions.
# This keeps the original list columns (ALCL11 and JOB), but adds binary indicators.

def _norm_dummy_label(x):
    """Normalize labels so dummies are robust to whitespace and quote variants."""
    if not isinstance(x, str):
        return x
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("’", "'").replace("‘", "'")
    x = x.replace("“", '"').replace("”", '"')
    x = " ".join(x.split())
    return x


def _as_list(value):
    """Return a list for list-like survey answers, preserving safe behavior for blanks."""
    if isinstance(value, list):
        return value
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    if isinstance(value, str):
        value = value.strip()
        if value == "":
            return []
        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                return parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                return [value]
    return [value]


def _add_multiselect_dummies(data_dict, source_col, dummy_map):
    """Add 0/1 dummies to data_dict based on whether each label is selected."""
    normalized_dummy_map = {
        dummy_col: _norm_dummy_label(label)
        for dummy_col, label in dummy_map.items()
    }

    selected_sets = [
        {_norm_dummy_label(v) for v in _as_list(value)}
        for value in data_dict[source_col]
    ]

    for dummy_col, normalized_label in normalized_dummy_map.items():
        data_dict[dummy_col] = [
            int(normalized_label in selected)
            for selected in selected_sets
        ]


# ALCL11: important local issue selections -> ALCL1_imp ... ALCL10_imp
ALCL11_dummy_map = {
    f"ALCL{i}_imp": label
    for i, label in enumerate(translation_dict["ALCL11"]["EN"], start=1)
}
_add_multiselect_dummies(survey_df_dict_ENG, "ALCL11", ALCL11_dummy_map)


# JOB: employment-status multi-select question -> stable job-status dummy variables
JOB_dummy_map = {
    "JOB_paid_work": "In paid work (employee, self-employed, working for your family business) or temporary absent.",
    "JOB_education": "In education (not paid for by employer), even if on vacation",
    "JOB_unemployed_looking": "Unemployed and actively looking for a job",
    "JOB_unemployed_not_looking": "Unemployed, wanting a job but not actively looking for a job",
    "JOB_permanently_sick_disabled": "Permanently sick or disabled",
    "JOB_retired": "Retired",
    "JOB_community_military_service": "In community or military service",
    "JOB_housework_care": "Doing housework, looking after children or other persons",
    "JOB_other": "Other",
    "JOB_prefer_not_to_say": "Prefer not to say",
}
_add_multiselect_dummies(survey_df_dict_ENG, "JOB", JOB_dummy_map)

print("Added ALCL11 dummy columns:", list(ALCL11_dummy_map.keys()))
print("Added JOB dummy columns:", list(JOB_dummy_map.keys()))

Added ALCL11 dummy columns: ['ALCL1_imp', 'ALCL2_imp', 'ALCL3_imp', 'ALCL4_imp', 'ALCL5_imp', 'ALCL6_imp', 'ALCL7_imp', 'ALCL8_imp', 'ALCL9_imp', 'ALCL10_imp']
Added JOB dummy columns: ['JOB_paid_work', 'JOB_education', 'JOB_unemployed_looking', 'JOB_unemployed_not_looking', 'JOB_permanently_sick_disabled', 'JOB_retired', 'JOB_community_military_service', 'JOB_housework_care', 'JOB_other', 'JOB_prefer_not_to_say']


In [10]:
df_survey_response = pd.DataFrame.from_dict(survey_df_dict); df_survey_response.head()

,userId,surveyDuration,AGE,SCREEN_ELIGIBILITY,SCREEN_RESIDENCE,NSEC_B,NSEC_BCRT,NSEC_P,NECN_B,NECN_BCRT,...,SSBS1,SSBS2,SQDR1,GENDER,EDUCATION,JOB,SRQV1,language,ATTN_CHK_PF,ATTN_CHK_FAIL_REASON
0,3a203770e837d765cd5f6f984a4f598c,406.388,57,Ja,Ja,7,6,5,6,6,...,7,5,5,Kvinde,PhD anden forskeruddannelse,"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give alle stemmer til ...,DA,FAIL,Zero correct matches
1,3a203770e80d24440b6379c848d401f9,463.966,77,Ja,Ja,7,8,8,10,10,...,10,10,0,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),[Pensioneret],Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
2,3a203770e7e805db58c55016db059137,859.743,41,Ja,Ja,6,10,2,4,10,...,5,6,6,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
3,3a203770e848ec257fce0316dd4cfd65,276.418,48,Ja,Ja,7,9,9,6,9,...,10,10,0,Kvinde,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",It should be allowed to give all votes to just...,DA,PASS,None
4,3a203770e7efbe8b69127c396a4165f4,341.944,44,Ja,Ja,10,10,0,9,10,...,10,9,1,Mand,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,PASS,None


## Find where there's data missing

In [11]:
def get_nan_columns_for_row(dataframe, row_index):
    """
    Identifies which columns contain NaN (missing) values for a given row position.
    
    Parameters:
    dataframe (pd.DataFrame): The source DataFrame.
    row_index (int): The integer positional index (iloc) of the row to check.
    
    Returns:
    list: A list of column names that contain NaN values in that row.
    """
    # 1. Safely extract the target row using .iloc
    try:
        row = dataframe.iloc[row_index]
    except IndexError:
        print(f"Error: Row index {row_index} is out of bounds.")
        return []
        
    # 2. Filter for columns where the value is missing (NaN)
    nan_columns = row[row.isna()].index.tolist()
    
    # 3. Print a quick summary for readability
    if nan_columns:
        print(f"Row {row_index} has NaNs in columns: {nan_columns}")
    else:
        print(f"Row {row_index} has no NaN values.")
        
    return nan_columns

## Replace all blank cells with nans (these will save as nans in a csv anyway)

In [12]:
df_survey_response.replace(r'^\s*$', np.nan, regex=True, inplace=True);

In [13]:
## Count number of nans
nan_rows_count = df_survey_response.isna().any(axis=1).sum()
print(f"Number of rows with NaNs: {nan_rows_count}")

Number of rows with NaNs: 717


In [14]:
# Find the indices where there are nans
nan_iloc_indices = np.where(df_survey_response.isna().any(axis=1))[0].tolist()
# print(nan_iloc_indices)
for idx in nan_iloc_indices:
    get_nan_columns_for_row(df_survey_response, idx)

Row 0 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 1 has NaNs in columns: ['VOTE_NAT_CNDT']
Row 3 has NaNs in columns: ['VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2', 'ATTN_CHK_FAIL_REASON']
Row 4 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 6 has NaNs in columns: ['VOTE_LOC_CNDT']
Row 9 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 10 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 11 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 14 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 15 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 18 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 20 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 23 has NaNs in columns: ['ATTN_CHK_FAIL_REASON']
Row 25 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_CNDT']
Row 26 has NaNs in co

## Drop column ATTN_CHK_FAIL_REASON because it has many blank entries
### *Minsu's additional Edit: Drop column SRQV1 as it's a measure of post-QV-experiment attitudes. 

In [15]:
df = df_survey_response.drop(columns=['ATTN_CHK_FAIL_REASON','SRQV1'])

In [16]:
nan_rows_count2 = df.isna().any(axis=1).sum()
print(f"Number of rows with NaNs: {nan_rows_count2}")

Number of rows with NaNs: 188


In [17]:
# Find the indices where there are nans
nan_iloc_indices2 = np.where(df.isna().any(axis=1))[0].tolist()
# print(nan_iloc_indices)
for idx in nan_iloc_indices2:
    get_nan_columns_for_row(df, idx)

Row 0 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 1 has NaNs in columns: ['VOTE_NAT_CNDT']
Row 3 has NaNs in columns: ['VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 6 has NaNs in columns: ['VOTE_LOC_CNDT']
Row 20 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 25 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_CNDT']
Row 30 has NaNs in columns: ['VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 42 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT', 'VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']
Row 43 has NaNs in columns: ['VOTE_LOC_CNDT', 'VOTE_NAT_CNDT']
Row 48 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT']
Row 49 has NaNs in columns: ['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT']
Row 52 has NaNs in columns: ['VOTE_LOC_PRTY', 

# Minsu's edit to preserve the following items:
['VOTE_LOC_PRTY', 'VOTE_LOC_CNDT', 'VOTE_NAT_PRTY', 'VOTE_NAT_CNDT']

In [27]:
# ============================================================
# Preserve observations by recoding structural missingness
# in vote party/candidate fields
# ============================================================

# ------------------------------------------------------------
# 1. Standardize blank strings as NaN
# ------------------------------------------------------------

df = df.replace(r"^\s*$", np.nan, regex=True)


# ------------------------------------------------------------
# 2. Define turnout labels
# ------------------------------------------------------------

VOTED_VALUES = {
    "I am sure I voted.",
    "Jeg er sikker på, at jeg stemte.",
}

DID_NOT_VOTE_LABEL = "Did not vote"


# ------------------------------------------------------------
# 3. Fill party/candidate fields for respondents who did not vote
# ------------------------------------------------------------

def fill_party_candidate_when_did_not_vote(
    data,
    turnout_col,
    party_col,
    candidate_col,
    voted_values=VOTED_VALUES,
    fill_value=DID_NOT_VOTE_LABEL
):
    """
    If respondent did not vote according to the turnout column,
    structurally empty party/candidate fields should be coded as 'Did not vote'.

    This only fills party/candidate cells that are currently missing.
    It does not overwrite non-missing responses.
    """

    did_not_vote = (
        data[turnout_col].notna()
        & ~data[turnout_col].isin(voted_values)
    )

    for col in [party_col, candidate_col]:
        data.loc[did_not_vote & data[col].isna(), col] = fill_value

    return data


df = fill_party_candidate_when_did_not_vote(
    data=df,
    turnout_col="VOTE_LOC",
    party_col="VOTE_LOC_PRTY",
    candidate_col="VOTE_LOC_CNDT"
)

df = fill_party_candidate_when_did_not_vote(
    data=df,
    turnout_col="VOTE_NAT",
    party_col="VOTE_NAT_PRTY",
    candidate_col="VOTE_NAT_CNDT"
)


In [28]:
# ------------------------------------------------------------
# 4. Identify missing or invalid candidate entries
# ------------------------------------------------------------

def is_invalid_candidate_entry(x):
    """
    Identify invalid candidate entries caused by the text-entry recording issue.

    Treat as invalid:
    - missing values
    - empty strings
    - single-character strings such as 'A', 'a', 'O', etc.
    - 'Blank' / 'Blank,' typed into the free-text entry box

    Do NOT treat meaningful responses such as:
    - 'I voted only for the party.'
    - 'I do not remember'
    - 'I don't remember'
    as invalid.
    """

    if pd.isna(x):
        return True

    if not isinstance(x, str):
        return False

    x_clean = x.strip()

    if x_clean == "":
        return True

    # Single-character entries are likely truncated free-text responses
    if len(x_clean) == 1:
        return True

    # Some free-text entries are recorded literally as "Blank" or "Blank,"
    # even though the respondent selected an Other-party option.
    # Treat these as invalid candidate names and recode them to '<party> _ Other'.
    if x_clean.strip("'\"`.,;: ").lower() == "blank":
        return True

    return False


# ------------------------------------------------------------
# 5. For reported voters, recode missing/single-letter/'Blank'
#    candidate entries as '<party> _ Other', while preserving
#    genuine 'Don't know' cases.
# ------------------------------------------------------------

def fill_invalid_candidate_with_party_other(
    data,
    turnout_col,
    party_col,
    candidate_col,
    voted_values=VOTED_VALUES,
    suffix="Other"
):
    """
    If respondent says they voted, reported a valid party, but the candidate
    entry is missing, single-character, or 'Blank', recode candidate as:
        '<party> _ Other'

    If the party field itself says "Don't know" / "I don't remember",
    preserve that as "Don't know" rather than creating values like:
        "Don't know _ Other"
    """

    unknown_party_values = {
        "Don't know",
        "I don't remember",
        "I do not remember",
    }

    non_party_values = unknown_party_values | {
        "Did not vote",
        "Voted, party not reported",
    }

    voted = data[turnout_col].isin(voted_values)
    invalid_candidate = data[candidate_col].apply(is_invalid_candidate_entry)

    # Case A: reported voting, but party is unknown and candidate is missing/invalid.
    # Preserve uncertainty as "Don't know" instead of "Don't know _ Other".
    unknown_party_mask = (
        voted
        & data[party_col].isin(unknown_party_values)
        & invalid_candidate
    )
    data.loc[unknown_party_mask, candidate_col] = "Don't know"

    # Case B: reported voting, valid party is available, and candidate is missing/invalid.
    # Recode as '<party> _ Other'. This handles missing values, one-letter entries,
    # and literal 'Blank' / 'Blank,' entries.
    has_valid_party = (
        data[party_col].notna()
        & ~data[party_col].isin(non_party_values)
    )

    mask = voted & has_valid_party & invalid_candidate

    data.loc[mask, candidate_col] = (
        data.loc[mask, party_col].astype(str) + " _ " + suffix
    )

    return data


df = fill_invalid_candidate_with_party_other(
    data=df,
    turnout_col="VOTE_LOC",
    party_col="VOTE_LOC_PRTY",
    candidate_col="VOTE_LOC_CNDT"
)

df = fill_invalid_candidate_with_party_other(
    data=df,
    turnout_col="VOTE_NAT",
    party_col="VOTE_NAT_PRTY",
    candidate_col="VOTE_NAT_CNDT"
)


# ------------------------------------------------------------
# 6. Cleanup any artifacts that may already have been generated
# ------------------------------------------------------------

for candidate_col in ["VOTE_LOC_CNDT", "VOTE_NAT_CNDT"]:
    if candidate_col in df.columns:
        df[candidate_col] = df[candidate_col].replace({
            "Don't know _ Other": "Don't know",
            "I don't remember _ Other": "I don't remember",
            "I do not remember _ Other": "I don't remember",
        })



In [29]:
# Find the indices where there are still nans
nan_iloc_indices2 = np.where(df.isna().any(axis=1))[0].tolist()
# print(nan_iloc_indices)
for idx in nan_iloc_indices2:
    get_nan_columns_for_row(df, idx)

# Minsu's edit on ['VCRT1', 'VCRT2', 'VRGT1', 'VRGT2']

NOTE: VCRT1, VCRT2, VRGT1, and VRGT2 ask about certainty/regret regarding the respondent's national-election choice. These questions are only applicable to respondents who voted in the national election. For national non-voters, missing values are structural, not accidental. For this, herer, I encode them as -99 in this GAN-training export to avoid dropping, otherwise meaningful non-voter observations. Valid substantive values are 0–10. -99 = Not applicable because the respondent did not vote nationally. 

You can change this if you need. 

In [30]:
# ------------------------------------------------------------
# Handle national-election certainty/regret variables for GAN export
# ------------------------------------------------------------

NATIONAL_VOTED_VALUES = {
    "I am sure I voted.",
    "Jeg er sikker på, at jeg stemte.",
}

certainty_regret_cols = ["VCRT1", "VCRT2", "VRGT1", "VRGT2"]

# Indicator: whether national certainty/regret questions are applicable
df["VOTE_NAT_DID_VOTE"] = df["VOTE_NAT"].isin(NATIONAL_VOTED_VALUES).astype(int)

# Ensure these columns are numeric
for col in certainty_regret_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# For national non-voters, these questions are structurally inapplicable.
# To prevent row deletion in GAN training/export, encode as -99.
df.loc[df["VOTE_NAT_DID_VOTE"] == 0, certainty_regret_cols] = -99

In [31]:
# Find the indices where there are still nans
nan_iloc_indices2 = np.where(df.isna().any(axis=1))[0].tolist()
# print(nan_iloc_indices)
for idx in nan_iloc_indices2:
    get_nan_columns_for_row(df, idx)

# these missing values can be dropped finally.

## Drop all voters who don't have complete data

In [32]:
df = df.drop(df.index[nan_iloc_indices2])
df.shape

(623, 65)

## Drop respondents who failed the attention check

The previous version also dropped English-language responses. I do not apply that filter here because the translation step has been revised to translate item values directly, rather than relying on the respondent-level language flag. Therefore, both Danish and English responses can be retained in the English-translated output.

If needed for a specific downstream purpose, English-language responses can still be excluded by uncommenting the language-filter line below.

In [33]:
df = df[df['ATTN_CHK_PF'] != 'FAIL']
#df = df[df['language'] != 'EN']

In [34]:
df.shape

(623, 65)

In [35]:
## Count number of nans
nan_rows_count3 = df.isna().any(axis=1).sum()
print(f"Number of rows with NaNs: {nan_rows_count3}")

Number of rows with NaNs: 0


## There are no more nans but there are still problems

## Unique values analysis

In [36]:
def column_unique_values(df, column_name):
    """
    Extracts and prints the unique values for a single specified column in a DataFrame.
    """
    # Quick safety check to make sure the column actually exists
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in the DataFrame.")
        return
    unique_values = df[column_name].unique().tolist()
    
    return unique_values

In [37]:
column_unique_values(df, 'VOTE_LOC_CNDT')

['I voted only for the party.',
 'Anders Winnerskjold',
 'Maja Albrechtsen',
 'C. Det Konservative Folkeparti _ Other',
 'Peter Sporleder',
 "I don't remember",
 'Thomas Medom',
 'V. Venstre, Danmarks Liberale Parti _ Other',
 'Did not vote',
 'Louise Svenstrup',
 'Anette Poulsen',
 'Jakob Søgaard Clausen',
 'Polly Bak Dutschke',
 'A. Socialdemokratiet _ Other',
 'Other',
 'Sarah Jarsbo',
 'Mette Bjerre',
 'Jesper Kjeldsen',
 'F. SF - Socialistisk Folkeparti _ Other',
 'Jeppe Spure Nielsen',
 'M. Moderaterne _ Other',
 'Tenna Røberg',
 'Metin Lindved Aydin',
 'Nanna Lippert Troelsen',
 "Don't know",
 'Isabella Heilmann',
 'Anna Thusgård',
 'Nikoline Erbs Hillers-Bendtsen',
 'Eva Pannerup',
 'Å. Alternativet _ Other',
 'Albert Rosenkrantz Conradsen',
 'Karina Bundgaard',
 'Mads Madsen',
 'Henrik Arens',
 'Else Kayser',
 'Thomas Kastrup Christensen',
 'Mahad B. Yussuf',
 'Sinisa Lozo',
 'Thor C. Jonasen',
 'Nicolaj Bang',
 'Annemarie Gottlieb',
 'Æ. Danmarksdemokraterne ‒ Inger Støjberg 

In [38]:
column_unique_values(df, 'VOTE_NAT_CNDT')

['Did not vote',
 'Nicolai Wammen',
 'Morten Siig Henriksen',
 'Mona Juul',
 'Peter Sporleder',
 'I voted only for the party.',
 'Lars Boje Mathiesen',
 'Alex Vanopslagh',
 'Maria Temponeras',
 'Nana Harring',
 'Sofie Lippert',
 "I don't remember",
 'Trine Mach',
 'Troels Lund Poulsen',
 'Kirsten Normann Andersen',
 'Katrine Robsøe',
 'Hanne Roed',
 'Louise Svenstrup',
 'Anna Brændemose',
 'Nick Zimmermann',
 'Christina Egelund',
 'Other',
 'Other _ Other',
 'Anders Kühnau',
 'Camilla Fabricius',
 'Paw Hedegaard Amdisen',
 'Jacob Skjærris',
 'Ø. Enhedslisten – De Rød-Grønne _ Other',
 'Jakob Jensen',
 'Jens Meilvang',
 'Æ. Danmarksdemokraterne ‒ Inger Støjberg _ Other',
 'Anne Sophie Callesen',
 'Caroline Stage Olsen',
 'Dorthe Hindborg',
 'Leif Lahn Jensen',
 'Line Rasmussen',
 'Mathilde Hjort Bressum',
 'Søren Lahn Sloth',
 'Dennis Munk',
 "Don't know",
 'A. Socialdemokratiet _ Other',
 'Charlotte Broman Mølbæk',
 'Lisbeth Ivanhoe',
 'Thor Clasen Jonasen',
 'Charlotte Vindeløv',
 'Er

# Harmonize Danish/English response labels into a target language

This step is intended to avoid treating substantively identical responses as different categories simply because they were recorded in different languages. In the raw data, some respondents answered in Danish while others answered in English, and a few fields contained mixed-language values. If these values are passed directly to the synthetic-data generation pipeline, the model may treat Danish and English versions of the same response as separate categories, which could introduce unnecessary measurement error.

To address this, the code below harmonizes all dictionary-covered survey response labels into a single target language. The function can produce either an English-harmonized or Danish-harmonized version of the dataset, so the downstream synthetic-data team can choose which version is more appropriate for their workflow.

This harmonization only applies to controlled survey response labels covered by the translation dictionary. Official names, such as party names, candidate names, and district or constituency names, are preserved unless they are explicitly included in the translation dictionary.

In [39]:
def normalize_text_for_matching(x):
    """
    Normalize unicode, apostrophes, quotes, and whitespace
    so that dictionary matching is more robust.
    """
    if not isinstance(x, str):
        return x

    x = unicodedata.normalize("NFKC", x)
    x = x.replace("’", "'").replace("‘", "'")
    x = x.replace("“", '"').replace("”", '"')
    x = " ".join(x.split())
    return x


def parse_list_if_needed(x):
    """
    Some multi-select columns are stored as Python lists,
    while others may be stored as string representations of lists.
    This function safely converts list-like strings back to lists.
    """
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return x

    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return x

    return x


def make_translation_maps(translation_dict, target_language="EN"):
    """
    Build a dictionary for each translated survey column.

    If target_language == "EN":
        Danish label -> English label
        English label -> English label

    If target_language == "DA":
        English label -> Danish label
        Danish label -> Danish label

    This means the function can be safely applied to mixed-language data.
    """

    if target_language not in {"EN", "DA"}:
        raise ValueError("target_language must be either 'EN' or 'DA'.")

    source_languages = ["DA", "EN"]
    translation_maps = {}

    for col, langs in translation_dict.items():
        da_vals = langs.get("DA", [])
        en_vals = langs.get("EN", [])

        if len(da_vals) != len(en_vals):
            raise ValueError(
                f"Translation list length mismatch for {col}: "
                f"{len(da_vals)} Danish values vs {len(en_vals)} English values"
            )

        target_vals = en_vals if target_language == "EN" else da_vals

        mapping = {}

        for da, en, target in zip(da_vals, en_vals, target_vals):
            # Danish -> target language
            mapping[da] = target
            mapping[normalize_text_for_matching(da)] = target

            # English -> target language
            mapping[en] = target
            mapping[normalize_text_for_matching(en)] = target

        translation_maps[col] = mapping

    return translation_maps


def translate_value_to_target_language(value, mapping):
    """
    Translate one value or a list of values into the target language.
    Unknown values are preserved.
    """
    value = parse_list_if_needed(value)

    if isinstance(value, list):
        return [translate_value_to_target_language(v, mapping) for v in value]

    if pd.isna(value):
        return value

    value_norm = normalize_text_for_matching(value)

    return mapping.get(value, mapping.get(value_norm, value))


def harmonize_dataframe_language(data, translation_dict, target_language="EN"):
    """
    Apply item-level language harmonization to all columns covered
    by translation_dict.

    This avoids treating Danish and English versions of the same
    survey response as separate categories in synthetic-data generation.
    """
    data = data.copy()
    translation_maps = make_translation_maps(
        translation_dict=translation_dict,
        target_language=target_language
    )

    for col, mapping in translation_maps.items():
        if col in data.columns:
            data[col] = data[col].apply(
                lambda x: translate_value_to_target_language(x, mapping)
            )

    return data

In [40]:
# English-harmonized version
df_eng = harmonize_dataframe_language(
    data=df,
    translation_dict=translation_dict,
    target_language="EN"
)

# Danish-harmonized version
df_da = harmonize_dataframe_language(
    data=df,
    translation_dict=translation_dict,
    target_language="DA"
)

In [41]:
# ============================================================
# QC checks for language harmonization
# ============================================================

def flatten_value_for_qc(value):
    """
    Convert a cell value into a list of values for QC.
    Handles normal strings, lists, and string-representations of lists.
    """
    value = parse_list_if_needed(value)

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return [value]


def check_remaining_source_language_labels(
    data,
    translation_dict,
    source_language="DA",
    target_language="EN",
    dataset_name="df_eng"
):
    """
    Check whether labels from the source language remain in a harmonized dataset.

    Example:
    - For English-harmonized data, source_language='DA'
      checks whether Danish dictionary labels remain.
    - For Danish-harmonized data, source_language='EN'
      checks whether English dictionary labels remain.
    """

    remaining = {}

    for col, langs in translation_dict.items():
        if col not in data.columns:
            continue

        source_labels = set(langs.get(source_language, []))
        source_labels_norm = {
            normalize_text_for_matching(x)
            for x in source_labels
        }

        bad_entries = []

        for row_idx, value in data[col].items():
            values = flatten_value_for_qc(value)

            for v in values:
                if not isinstance(v, str):
                    continue

                v_norm = normalize_text_for_matching(v)

                if v in source_labels or v_norm in source_labels_norm:
                    bad_entries.append({
                        "row_index": row_idx,
                        "column": col,
                        "value": v
                    })

        if bad_entries:
            remaining[col] = bad_entries

    print(f"\nQC: Remaining {source_language} labels in {dataset_name}")
    print("=" * 70)

    if not remaining:
        print(f"PASS: No dictionary-covered {source_language} labels remain in {dataset_name}.")
    else:
        print(f"WARNING: Some dictionary-covered {source_language} labels remain in {dataset_name}.")
        for col, entries in remaining.items():
            print(f"\n{col}: {len(entries)} problematic entries")
            for entry in entries[:10]:
                print(entry)
            if len(entries) > 10:
                print(f"... {len(entries) - 10} more")

    return remaining


def check_unexpected_translated_values(
    data,
    translation_dict,
    target_language="EN",
    dataset_name="df_eng"
):
    """
    Check for values in dictionary-covered columns that are not part of the
    expected target-language dictionary.

    Important:
    This will flag true free-text or official-name values if the column contains
    values not covered by the dictionary. Therefore, this QC is diagnostic,
    not automatically an error.
    """

    unexpected = {}

    for col, langs in translation_dict.items():
        if col not in data.columns:
            continue

        expected_labels = set(langs.get(target_language, []))
        expected_labels_norm = {
            normalize_text_for_matching(x)
            for x in expected_labels
        }

        bad_entries = []

        for row_idx, value in data[col].items():
            values = flatten_value_for_qc(value)

            for v in values:
                if not isinstance(v, str):
                    continue

                v_norm = normalize_text_for_matching(v)

                if v not in expected_labels and v_norm not in expected_labels_norm:
                    bad_entries.append({
                        "row_index": row_idx,
                        "column": col,
                        "value": v
                    })

        if bad_entries:
            unexpected[col] = bad_entries

    print(f"\nQC: Unexpected values in dictionary-covered columns of {dataset_name}")
    print("=" * 70)

    if not unexpected:
        print(f"PASS: All non-missing values in dictionary-covered columns match the {target_language} dictionary.")
    else:
        print(
            "NOTE: Some values are not in the target-language dictionary. "
            "This may be fine if they are free-text responses, official names, "
            "or values intentionally left unchanged."
        )
        for col, entries in unexpected.items():
            print(f"\n{col}: {len(entries)} unexpected entries")
            unique_examples = sorted({str(e["value"]) for e in entries})[:20]
            for val in unique_examples:
                print(f"  - {val}")
            if len(unique_examples) == 20:
                print("  ...")

    return unexpected


def summarize_language_qc_results(remaining_source_labels, unexpected_values, dataset_name):
    """
    Print compact summary for notebook readability.
    """
    n_remaining = sum(len(v) for v in remaining_source_labels.values())
    n_unexpected = sum(len(v) for v in unexpected_values.values())

    print(f"\nSummary for {dataset_name}")
    print("=" * 70)
    print(f"Remaining source-language dictionary labels: {n_remaining}")
    print(f"Unexpected values in dictionary-covered columns: {n_unexpected}")

    if n_remaining == 0:
        print("Main translation QC: PASS")
    else:
        print("Main translation QC: CHECK REQUIRED")


# ------------------------------------------------------------
# Run QC for English-harmonized dataset
# ------------------------------------------------------------

eng_remaining_da = check_remaining_source_language_labels(
    data=df_eng,
    translation_dict=translation_dict,
    source_language="DA",
    target_language="EN",
    dataset_name="df_eng"
)

eng_unexpected = check_unexpected_translated_values(
    data=df_eng,
    translation_dict=translation_dict,
    target_language="EN",
    dataset_name="df_eng"
)

summarize_language_qc_results(
    remaining_source_labels=eng_remaining_da,
    unexpected_values=eng_unexpected,
    dataset_name="df_eng"
)


# ------------------------------------------------------------
# Run QC for Danish-harmonized dataset
# ------------------------------------------------------------

da_remaining_en = check_remaining_source_language_labels(
    data=df_da,
    translation_dict=translation_dict,
    source_language="EN",
    target_language="DA",
    dataset_name="df_da"
)

da_unexpected = check_unexpected_translated_values(
    data=df_da,
    translation_dict=translation_dict,
    target_language="DA",
    dataset_name="df_da"
)

summarize_language_qc_results(
    remaining_source_labels=da_remaining_en,
    unexpected_values=da_unexpected,
    dataset_name="df_da"
)


QC: Remaining DA labels in df_eng
PASS: No dictionary-covered DA labels remain in df_eng.

QC: Unexpected values in dictionary-covered columns of df_eng
PASS: All non-missing values in dictionary-covered columns match the EN dictionary.

Summary for df_eng
Remaining source-language dictionary labels: 0
Unexpected values in dictionary-covered columns: 0
Main translation QC: PASS

QC: Remaining EN labels in df_da
PASS: No dictionary-covered EN labels remain in df_da.

QC: Unexpected values in dictionary-covered columns of df_da
PASS: All non-missing values in dictionary-covered columns match the DA dictionary.

Summary for df_da
Remaining source-language dictionary labels: 0
Unexpected values in dictionary-covered columns: 0
Main translation QC: PASS


## Save to mac compatible encoding

In [42]:
df_eng.to_csv("survey_responses_GAN_ENG.csv", encoding="utf-8-sig", index=False)
df_da.to_csv("survey_responses_GAN_DA.csv", encoding="utf-8-sig", index=False)